
# Predictive Vehicle Maintenance System

## Machine Learning Pipeline

### 1. Data Collection Sources
- Car sensor data via On-board diagnostics (OBD-II)
- Historical vehicle breakdown data
- Kaggle Datasets (OBD-II datasets)
  - Features: 33 columns representing various vehicle sensor data

### 2. Key Predictive Models

#### Model 1: Failure Probability Prediction
- **Type**: Binary Classification
- **Algorithm**: Random Forest Classifier
- **Classification Categories**:
  - Normal Case
  - Problem Cases

#### Model 2: Trouble Code Classification
- **Type**: Multi-Classification
- **Algorithm**: Random Forest Classifier
- **Purpose**: Classify between 12 types of trouble codes
- **Key Question**: What is the probability of a failure occurring within the next X hours?

#### Model 3: Remaining Useful Life Prediction
- **Type**: Regression
- **Algorithm**: ExtraTreesRegressor
- **Purpose**: Predict remaining useful life of vehicle components

#### Model 4: Service Priority Assessment
- **Type**: Natural Language Processing
- **Model**: GPT-4
- **Key Question**: Which asset requires servicing most urgently?

### 3. Implementation
- Testing and deployment via Streamlit
- Handles imbalanced data scenarios
- Focus on model performance optimization



# Vehicle Data Processing Pipeline

## Overview

The `VehicleDataProcessor` class provided processes and analyzes vehicle diagnostic data by performing preprocessing steps such as data cleaning, feature transformation, and filtering. Below is a breakdown of its functionality and how it can be used:

### Features of `VehicleDataProcessor`
1. **Initialization**:
   - Accepts a CSV file path containing vehicle diagnostic data and reads it into a pandas DataFrame.

2. **Preprocessing (`preprocess_data`)**:
   - Cleans and transforms the dataset:
     - Fills missing values for `TROUBLE_CODES` with "normal" under specific conditions.
     - Drops rows with missing `TROUBLE_CODES`.
     - Removes unnecessary columns.
     - Converts relevant columns to numeric formats.
     - Handles missing values with mean imputation and interpolation.

3. **Cleaning Numeric Columns (`_clean_numeric_columns`)**:
   - Processes specific columns such as `SHORT TERM FUEL TRIM BANK 1`, `ENGINE_LOAD`, and others to ensure numeric consistency.

4. **Handling Missing Values (`_handle_missing_values`)**:
   - Fills missing values in numeric columns with their mean and interpolates other missing values.

5. **Filter for Specific Trouble Codes (`filter_trouble_codes`)**:
   - Extracts data rows corresponding to a specific trouble code.

---


In [ ]:
import numpy as np
import pandas as pd
import os
from google.colab import drive
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import (
    RandomForestClassifier,
    RandomForestRegressor,
    ExtraTreesRegressor
)
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Mount Google Drive
drive.mount('/content/drive')

# Define file path
file_path = '/content/drive/My Drive/capstone/exp1_14drivers_14cars_dailyRoutes.csv'

class VehicleDataProcessor:
    """
    A class to process and analyze vehicle diagnostic data.
    """

    def __init__(self, file_path):
        """
        Initialize the data processor with the file path.

        Parameters:
        -----------
        file_path : str
            Path to the CSV file containing vehicle data
        """
        self.data = pd.read_csv(file_path)
        self.processed_data = None

    def preprocess_data(self):
        """
        Perform all data preprocessing steps including cleaning and transforming.
        """
        # Create a copy of the original data
        self.processed_data = self.data.copy()

        # Handle DTC_NUMBER and TROUBLE_CODES
        mask = (self.processed_data['DTC_NUMBER'] == 'MIL is OFF0 codes') & \
               (self.processed_data['TROUBLE_CODES'].isna())
        self.processed_data.loc[mask, 'TROUBLE_CODES'] = 'normal'

        # Drop rows with missing TROUBLE_CODES
        self.processed_data.dropna(subset=['TROUBLE_CODES'], inplace=True)

        # Remove unnecessary columns
        columns_to_drop = [
            'TIMESTAMP', 'DTC_NUMBER', 'AUTOMATIC', 'EQUIV_RATIO', 'ENGINE_RUNTIME',
            'FUEL_TYPE', 'SHORT TERM FUEL TRIM BANK 2', 'FUEL_PRESSURE',
            'LONG TERM FUEL TRIM BANK 2', 'MAF', 'AMBIENT_AIR_TEMP', 'FUEL_LEVEL',
            'BAROMETRIC_PRESSURE(KPA)', 'VEHICLE_ID', 'MODEL', 'YEAR', 'MARK',
            'MONTHS', 'DAYS_OF_WEEK', 'MIN', 'VEHICLE_ID', 'INTAKE_MANIFOLD_PRESSURE',
            'CAR_YEAR'
        ]
        self.processed_data = self.processed_data.drop(columns_to_drop, axis=1)

        # Clean and convert numeric columns
        self._clean_numeric_columns()

        # Handle missing values
        self._handle_missing_values()

        return self.processed_data

    def _clean_numeric_columns(self):
        """
        Clean and convert numeric columns to proper format.
        """
        # Clean SHORT TERM FUEL TRIM BANK 1
        self.processed_data['SHORT TERM FUEL TRIM BANK 1'] = (
            self.processed_data['SHORT TERM FUEL TRIM BANK 1']
            .str.split("%")
            .str.get(0)
            .astype(float)
        )

        # Clean ENGINE_LOAD
        self.processed_data['ENGINE_LOAD'] = (
            self.processed_data['ENGINE_LOAD']
            .str.split("%")
            .str.get(0)
            .str.replace(',', '.')
            .astype(float)
        )

        # Clean THROTTLE_POS
        self.processed_data['THROTTLE_POS'] = (
            self.processed_data['THROTTLE_POS']
            .str.split("%")
            .str.get(0)
            .astype(float)
        )

        # Clean TIMING_ADVANCE
        self.processed_data['TIMING_ADVANCE'] = (
            self.processed_data['TIMING_ADVANCE']
            .str.split("%")
            .str.get(0)
            .str.replace(',', '.')
            .astype(float)
        )

        # Clean ENGINE_POWER
        engine_power_mapping = {'1,6': 1.6, '1,8': 1.8, '1,4': 1.4}
        self.processed_data['ENGINE_POWER'] = (
            self.processed_data['ENGINE_POWER']
            .replace(engine_power_mapping)
            .astype(float)
        )

    def _handle_missing_values(self):
        """
        Handle missing values in the dataset using mean imputation.
        """
        numeric_columns = [
            'SHORT TERM FUEL TRIM BANK 1',
            'ENGINE_POWER',
            'ENGINE_LOAD',
            'THROTTLE_POS',
            'TIMING_ADVANCE'
        ]

        for column in numeric_columns:
            mean_value = self.processed_data[column].mean()
            self.processed_data[column].fillna(mean_value, inplace=True)

        # Interpolate remaining missing values
        self.processed_data = self.processed_data.interpolate()

    def filter_trouble_codes(self, trouble_code):
        """
        Filter data for specific trouble codes.

        Parameters:
        -----------
        trouble_code : str
            The trouble code to filter for

        Returns:
        --------
        pd.DataFrame
            Filtered dataset containing only the specified trouble code
        """
        return self.processed_data[self.processed_data['TROUBLE_CODES'] == trouble_code]

# Usage
if __name__ == "__main__":
    # Initialize processor
    processor = VehicleDataProcessor(file_path)

    # Process data
    cleaned_data = processor.preprocess_data()
    data = cleaned_data.copy()
    data_1 = data.copy()
    data_2 = data.copy()

    # Filter for specific trouble codes
    filtered_data = processor.filter_trouble_codes("P0079P2004P3000")

    print("Data shape:", cleaned_data.shape)
    print("\nMissing values:\n", cleaned_data.isna().sum())
    print("\nFiltered data head:\n", filtered_data.head())


Mounted at /content/drive


<ipython-input-2-12502453056d>:41: DtypeWarning: Columns (1,2,4,5,6,9,10,14,15,16,20,21,22,23,24,25,26,27) have mixed types. Specify dtype option on import or set low_memory=False.
  self.data = pd.read_csv(file_path)


Data shape: (47139, 11)

Missing values:
 ENGINE_POWER                   0
ENGINE_COOLANT_TEMP            0
ENGINE_LOAD                    0
ENGINE_RPM                     0
AIR_INTAKE_TEMP                0
SPEED                          0
SHORT TERM FUEL TRIM BANK 1    0
THROTTLE_POS                   0
TROUBLE_CODES                  0
TIMING_ADVANCE                 0
HOURS                          0
dtype: int64

Filtered data head:
        ENGINE_POWER  ENGINE_COOLANT_TEMP  ENGINE_LOAD  ENGINE_RPM  \
45310           1.4                 55.0    40.439026      2424.0   
45354           1.4                 91.0    86.300000      2725.0   
45554           1.4                 62.0    40.439026       819.0   
45568           1.4                 83.0    44.700000      2866.0   
45679           1.4                 92.0    36.900000       783.0   

       AIR_INTAKE_TEMP  SPEED  SHORT TERM FUEL TRIM BANK 1  THROTTLE_POS  \
45310             29.0   49.0                          2.0     26.000

<ipython-input-2-12502453056d>:138: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  self.processed_data[column].fillna(mean_value, inplace=True)
<ipython-input-2-12502453056d>:141: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  self.processed_data = self.processed_data.interpolate()


### Key Features of the `BinaryClassificationProcessor` Class

1. **Data Preparation**:
   - The class processes a DataFrame (`data`) containing vehicle information, including `TROUBLE_CODES`, which are encoded into binary values (`T_C` column).
   - The `prepare_target` method converts `TROUBLE_CODES` into binary form, where `normal` is mapped to `0` and any other value to `1`.

2. **Dataset Balancing**:
   - The `balance_dataset` method performs undersampling to balance the dataset, particularly addressing cases where the `normal` category might dominate. It ensures a balanced distribution for effective model training.

3. **Feature Preparation**:
   - The `prepare_features` method separates features and the target variable (`T_C`). It allows for dropping unnecessary or user-specified columns.

4. **Train-Test Split**:
   - The `split_data` method divides the dataset into training and testing subsets, ensuring reproducibility with a specified random state.

5. **Model Training**:
   - The `train_model` method trains a Random Forest classifier with customizable parameters.

6. **Model Evaluation**:
   - The `evaluate_model` method assesses the model's performance on the training and/or testing dataset, printing metrics such as accuracy, classification report, and confusion matrix.

7. **Model Saving and Loading**:
   - The `save_model` and `load_model` methods enable saving and loading the trained model using Python's `pickle` module.

8. **Predictions**:
   - The `predict` method generates predictions for new data using the trained model.

9. **Example Usage**:
   - Demonstrates how to use the class to prepare data, train the model, evaluate its performance, and save it for future use.

---






In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    confusion_matrix
)
import pickle
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

class BinaryClassificationProcessor:
    """
    A class to handle binary classification processing for vehicle state.
    """

    def __init__(self, data):
        """
        Initialize the binary classification processor.

        Parameters:
        -----------
        data : pd.DataFrame
            Preprocessed vehicle data
        """
        self.data = data.copy()
        self.X_train = None
        self.X_test = None
        self.y_train = None
        self.y_test = None
        self.model = None

    def prepare_target(self):
        """
        Prepare the target variable by encoding trouble codes.
        """
        # Display initial distribution of trouble codes
        print("Initial TROUBLE_CODES distribution:")
        print(self.data['TROUBLE_CODES'].value_counts())
        print("\n")

        # Encode trouble codes as binary (0 for normal, 1 for issues)
        self.data['T_C'] = self.data['TROUBLE_CODES'].apply(
            lambda x: 0 if x == 'normal' else 1
        )

        print("Binary encoded distribution:")
        print(self.data['T_C'].value_counts())
        print("\n")

    def balance_dataset(self, normal_sample_size=23214, random_state=42):
        """
        Balance the dataset using undersampling.

        Parameters:
        -----------
        normal_sample_size : int
            Number of normal cases to keep
        random_state : int
            Random state for reproducibility
        """
        # Undersample the majority class (normal cases)
        normal_indices = self.data[
            self.data['TROUBLE_CODES'] == 'normal'
        ].sample(n=normal_sample_size, random_state=random_state).index

        self.data = self.data.drop(normal_indices)

        print("Balanced dataset distribution:")
        print(self.data['T_C'].value_counts())
        print("\n")

    def prepare_features(self, additional_drops=None):
        """
        Prepare features for modeling.

        Parameters:
        -----------
        additional_drops : list
            Additional columns to drop from features
        """
        # Default columns to drop
        drops = ['T_C', 'TROUBLE_CODES', 'HOURS']

        # Add any additional columns to drop
        if additional_drops:
            drops.extend(additional_drops)

        # Prepare features and target
        self.X = self.data.drop(drops, axis=1)
        self.y = self.data['T_C']

    def split_data(self, test_size=0.2, random_state=42):
        """
        Split the data into training and testing sets.

        Parameters:
        -----------
        test_size : float
            Proportion of dataset to include in the test split
        random_state : int
            Random state for reproducibility
        """
        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
            self.X, self.y, test_size=test_size, random_state=random_state
        )

    def train_model(self, **model_params):
        """
        Train the Random Forest model.

        Parameters:
        -----------
        **model_params : dict
            Parameters to pass to RandomForestClassifier
        """
        self.model = RandomForestClassifier(**model_params)
        self.model.fit(self.X_train, self.y_train)

    def evaluate_model(self, dataset='both'):
        """
        Evaluate the model performance.

        Parameters:
        -----------
        dataset : str
            Which dataset to evaluate ('train', 'test', or 'both')
        """
        def get_metrics(X, y, dataset_name):
            pred = self.model.predict(X)
            clf_report = pd.DataFrame(
                classification_report(y, pred, output_dict=True)
            )

            print(f"{dataset_name} Results:")
            print("=" * 50)
            print(f"Accuracy Score: {accuracy_score(y, pred) * 100:.2f}%")
            print("_" * 50)
            print(f"Classification Report:\n{clf_report}")
            print("_" * 50)
            print(f"Confusion Matrix:\n{confusion_matrix(y, pred)}\n")

        if dataset.lower() in ['train', 'both']:
            get_metrics(self.X_train, self.y_train, "Training")

        if dataset.lower() in ['test', 'both']:
            get_metrics(self.X_test, self.y_test, "Testing")

    def save_model(self, filename='binary_model.sav'):
        """
        Save the trained model to disk.

        Parameters:
        -----------
        filename : str
            Name of the file to save the model
        """
        pickle.dump(self.model, open(filename, 'wb'))
        print(f"Model saved as {filename}")

    def load_model(self, filename='binary_model.sav'):
        """
        Load a trained model from disk.

        Parameters:
        -----------
        filename : str
            Name of the file containing the saved model
        """
        self.model = pickle.load(open(filename, 'rb'))
        print(f"Model loaded from {filename}")

    def predict(self, X):
        """
        Make predictions using the trained model.

        Parameters:
        -----------
        X : pd.DataFrame
            Features to make predictions on

        Returns:
        --------
        np.array
            Predicted classes
        """
        return self.model.predict(X)

# usage
if __name__ == "__main__":
    # Assuming 'data' is your preprocessed DataFrame
    classifier = BinaryClassificationProcessor(data)

    # Prepare data
    classifier.prepare_target()
    classifier.balance_dataset()
    classifier.prepare_features()
    classifier.split_data()

    # Train and evaluate model
    classifier.train_model(random_state=42)
    classifier.evaluate_model(dataset='both')

    # Save model
    classifier.save_model()

    # Make predictions on new data
    # new_predictions = classifier.predict(new_data)

Initial TROUBLE_CODES distribution:
TROUBLE_CODES
normal             35214
P0133               6070
C0300               5673
P0079P2004P3000       48
P0079C1004P3000       30
P0078U1004P3000       29
P007EP2036P18D0       20
P007EP2036P18E0       18
P0078B0004P3000       12
P007EP2036P18F0        9
P0079P1004P3000        5
P007FP2036P18E0        5
P007FP2036P18D0        3
P007FP2036P18F0        3
Name: count, dtype: int64


Binary encoded distribution:
T_C
0    35214
1    11925
Name: count, dtype: int64


Balanced dataset distribution:
T_C
0    12000
1    11925
Name: count, dtype: int64


Training Results:
Accuracy Score: 100.00%
__________________________________________________
Classification Report:
                0       1  accuracy  macro avg  weighted avg
precision     1.0     1.0       1.0        1.0           1.0
recall        1.0     1.0       1.0        1.0           1.0
f1-score      1.0     1.0       1.0        1.0           1.0
support    9613.0  9527.0       1.0    19140

This script defines a `CGANDataGenerator` class, which uses Conditional GANs to generate synthetic data for oversampling and balancing a dataset containing vehicle-related metrics, including trouble codes and hours. Here's a breakdown of its functionality:

### Key Features
1. **Data Preprocessing**:
   - Scales continuous features using `StandardScaler`.
   - Encodes categorical labels (`TROUBLE_CODES`) using `OneHotEncoder`.

2. **CGAN Model Architecture**:
   - **Generator**: Generates synthetic data based on noise and encoded trouble codes.
   - **Discriminator**: Evaluates the authenticity of the input data.
   - **CGAN**: Combines the generator and discriminator into a conditional GAN for training.

3. **Training Process**:
   - Trains the generator and discriminator alternately.
   - Ensures the discriminator learns to distinguish real from synthetic data while the generator improves to produce realistic data.

4. **Synthetic Data Generation**:
   - After training, the generator can create synthetic samples conditioned on specific trouble codes.


In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, LeakyReLU, BatchNormalization, Input, concatenate
from tensorflow.keras.optimizers import Adam
import tensorflow as tf

class CGANDataGenerator:
    """
    A class to handle data generation using Conditional GAN for oversampling vehicle data.
    """

    def __init__(self, data, trouble_code_col='TROUBLE_CODES', hours_col='HOURS'):
        """
        Initialize the CGAN data generator.

        Parameters:
        -----------
        data : pd.DataFrame
            Input data containing features and trouble codes
        trouble_code_col : str
            Name of the trouble codes column
        hours_col : str
            Name of the hours column
        """
        self.data = data.copy()
        self.trouble_code_col = trouble_code_col
        self.hours_col = hours_col

        self.scaler = StandardScaler()
        self.encoder = OneHotEncoder(sparse_output=False)

        self.generator = None
        self.discriminator = None
        self.cgan = None

        self.latent_dim = 100
        self.features_scaled = None
        self.trouble_codes_encoded = None
        self.combined_data = None

    def preprocess_data(self):
        """
        Preprocess the data by scaling features and encoding trouble codes.
        """
        # Separate features and targets
        self.features = self.data.drop(columns=[self.trouble_code_col, self.hours_col])
        self.trouble_codes = self.data[self.trouble_code_col]
        self.hours = self.data[self.hours_col]

        # Scale features
        self.features_scaled = self.scaler.fit_transform(self.features)

        # Encode trouble codes
        self.trouble_codes_encoded = self.encoder.fit_transform(
            self.trouble_codes.values.reshape(-1, 1)
        )

        # Combine data
        self.combined_data = np.hstack((
            self.features_scaled,
            self.trouble_codes_encoded,
            self.hours.values.reshape(-1, 1)
        ))

        self.num_classes = self.trouble_codes_encoded.shape[1]
        self.num_features = self.combined_data.shape[1]

    def build_generator(self):
        """
        Build the generator model.
        """
        noise_input = Input(shape=(self.latent_dim,))
        label_input = Input(shape=(self.num_classes,))
        merged_input = concatenate([noise_input, label_input])

        x = Dense(128)(merged_input)
        x = LeakyReLU(alpha=0.01)(x)
        x = BatchNormalization(momentum=0.8)(x)

        x = Dense(256)(x)
        x = LeakyReLU(alpha=0.01)(x)
        x = BatchNormalization(momentum=0.8)(x)

        x = Dense(self.num_features)(x)

        self.generator = Model([noise_input, label_input], x, name='Generator')

    def build_discriminator(self):
        """
        Build the discriminator model.
        """
        data_input = Input(shape=(self.num_features,))
        label_input = Input(shape=(self.num_classes,))
        merged_input = concatenate([data_input, label_input])

        x = Dense(512)(merged_input)
        x = LeakyReLU(alpha=0.01)(x)

        x = Dense(256)(x)
        x = LeakyReLU(alpha=0.01)(x)

        x = Dense(128)(x)
        x = LeakyReLU(alpha=0.01)(x)

        output = Dense(1, activation='sigmoid')(x)

        self.discriminator = Model([data_input, label_input], output, name='Discriminator')
        self.discriminator.compile(
            loss='binary_crossentropy',
            optimizer=Adam(learning_rate=0.0002, beta_1=0.5),
            metrics=['accuracy']
        )

    def build_cgan(self):
        """
        Build the combined CGAN model.
        """
        self.discriminator.trainable = False

        noise_input = Input(shape=(self.latent_dim,))
        label_input = Input(shape=(self.num_classes,))

        gen_data = self.generator([noise_input, label_input])
        validity = self.discriminator([gen_data, label_input])

        self.cgan = Model([noise_input, label_input], validity, name='CGAN')
        self.cgan.compile(
            loss='binary_crossentropy',
            optimizer=Adam(learning_rate=0.0002, beta_1=0.5)
        )

    def train(self, epochs=10000, batch_size=32, verbose=True):
        """
        Train the CGAN model.

        Parameters:
        -----------
        epochs : int
            Number of training epochs
        batch_size : int
            Size of training batches
        verbose : bool
            Whether to print training progress
        """
        valid = np.ones((batch_size, 1))
        fake = np.zeros((batch_size, 1))

        for epoch in range(epochs):
            # Train Discriminator
            idx = np.random.randint(0, self.combined_data.shape[0], batch_size)
            real_data = self.combined_data[idx]
            real_labels = self.trouble_codes_encoded[idx]

            noise = np.random.normal(0, 1, (batch_size, self.latent_dim))
            gen_data = self.generator.predict([noise, real_labels], verbose=0)

            d_loss_real = self.discriminator.train_on_batch(
                [real_data, real_labels], valid
            )
            d_loss_fake = self.discriminator.train_on_batch(
                [gen_data, real_labels], fake
            )
            d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)

            # Train Generator
            noise = np.random.normal(0, 1, (batch_size, self.latent_dim))
            g_loss = self.cgan.train_on_batch([noise, real_labels], valid)

            if verbose and epoch % 100 == 0:
                print(
                    f"Epoch {epoch}/{epochs} "
                    f"[D loss: {d_loss[0]:.4f}, acc: {100*d_loss[1]:.2f}%] "
                    f"[G loss: {g_loss:.4f}]"
                )

    def generate_synthetic_data(self, num_samples):
        """
        Generate synthetic data using the trained generator.

        Parameters:
        -----------
        num_samples : int
            Number of synthetic samples to generate

        Returns:
        --------
        pd.DataFrame
            DataFrame containing the generated synthetic data
        """
        # Generate noise and sample labels
        noise = np.random.normal(0, 1, (num_samples, self.latent_dim))
        sampled_labels = np.random.choice(
            self.encoder.categories_[0],
            num_samples
        )
        sampled_labels = self.encoder.transform(sampled_labels.reshape(-1, 1))

        # Generate synthetic data
        synthetic_data = self.generator.predict([noise, sampled_labels], verbose=0)

        # Separate features and create DataFrame
        synthetic_trouble_codes = self.encoder.inverse_transform(sampled_labels)
        synthetic_hours = synthetic_data[:, -1]
        synthetic_features = synthetic_data[:, :-self.num_classes-1]

        # Create DataFrame with original feature names
        synthetic_df = pd.DataFrame(
            self.scaler.inverse_transform(synthetic_features),
            columns=self.features.columns
        )
        synthetic_df[self.trouble_code_col] = synthetic_trouble_codes.flatten()
        synthetic_df[self.hours_col] = synthetic_hours

        return synthetic_df

# usage
if __name__ == "__main__":
    # Initialize generator
    data_generator = CGANDataGenerator(data_2)

    # Preprocess data
    data_generator.preprocess_data()

    # Build and train models
    data_generator.build_generator()
    data_generator.build_discriminator()
    data_generator.build_cgan()

    # Train the CGAN
    data_generator.train(epochs=10000, batch_size=32)

    # Generate synthetic data
    synthetic_data = data_generator.generate_synthetic_data(num_samples=2600)

    print("Original data shape:", data_generator.data.shape)
    print("Synthetic data shape:", synthetic_data.shape)

    # Combine original and synthetic data if needed
    combined_data = pd.concat([data_generator.data, synthetic_data], axis=0)

Epoch 0/10000 [D loss: 0.8604, acc: 25.00%] [G loss: 0.8611]
Epoch 100/10000 [D loss: 0.7757, acc: 45.66%] [G loss: 0.8122]
Epoch 200/10000 [D loss: 0.7953, acc: 38.42%] [G loss: 0.7644]
Epoch 300/10000 [D loss: 0.8168, acc: 30.85%] [G loss: 0.7225]
Epoch 400/10000 [D loss: 0.8451, acc: 24.50%] [G loss: 0.6771]
Epoch 500/10000 [D loss: 0.8853, acc: 19.96%] [G loss: 0.6284]
Epoch 600/10000 [D loss: 0.9344, acc: 16.75%] [G loss: 0.5798]
Epoch 700/10000 [D loss: 0.9953, acc: 14.40%] [G loss: 0.5342]
Epoch 800/10000 [D loss: 1.0613, acc: 12.63%] [G loss: 0.4932]
Epoch 900/10000 [D loss: 1.1292, acc: 11.25%] [G loss: 0.4565]
Epoch 1000/10000 [D loss: 1.1985, acc: 10.13%] [G loss: 0.4239]
Epoch 1100/10000 [D loss: 1.2670, acc: 9.25%] [G loss: 0.3955]
Epoch 1200/10000 [D loss: 1.3339, acc: 8.49%] [G loss: 0.3704]
Epoch 1300/10000 [D loss: 1.3968, acc: 7.85%] [G loss: 0.3478]
Epoch 1400/10000 [D loss: 1.4600, acc: 7.31%] [G loss: 0.3273]
Epoch 1500/10000 [D loss: 1.5194, acc: 6.83%] [G loss: 0

In [ ]:
synthetic_data.to_csv('synthetic_data.csv', index=False)

In [ ]:
synthetic_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2600 entries, 0 to 2599
Data columns (total 11 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   ENGINE_POWER                 2600 non-null   float32
 1   ENGINE_COOLANT_TEMP          2600 non-null   float32
 2   ENGINE_LOAD                  2600 non-null   float32
 3   ENGINE_RPM                   2600 non-null   float32
 4   AIR_INTAKE_TEMP              2600 non-null   float32
 5   SPEED                        2600 non-null   float32
 6   SHORT TERM FUEL TRIM BANK 1  2600 non-null   float32
 7   THROTTLE_POS                 2600 non-null   float32
 8   TIMING_ADVANCE               2600 non-null   float32
 9   TROUBLE_CODES                2600 non-null   object 
 10  HOURS                        2600 non-null   float32
dtypes: float32(10), object(1)
memory usage: 122.0+ KB


In [ ]:
synthetic_data['TROUBLE_CODES'].unique()

array(['C0300', 'P007FP2036P18F0', 'P007EP2036P18F0', 'normal',
       'P007FP2036P18D0', 'P0079P2004P3000', 'P0078B0004P3000',
       'P0079C1004P3000', 'P0133', 'P007FP2036P18E0', 'P0079P1004P3000',
       'P0078U1004P3000', 'P007EP2036P18E0', 'P007EP2036P18D0'],
      dtype=object)

In [ ]:
combined_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 49739 entries, 0 to 2599
Data columns (total 11 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   ENGINE_POWER                 49739 non-null  float64
 1   ENGINE_COOLANT_TEMP          49739 non-null  float64
 2   ENGINE_LOAD                  49739 non-null  float64
 3   ENGINE_RPM                   49739 non-null  float64
 4   AIR_INTAKE_TEMP              49739 non-null  float64
 5   SPEED                        49739 non-null  float64
 6   SHORT TERM FUEL TRIM BANK 1  49739 non-null  float64
 7   THROTTLE_POS                 49739 non-null  float64
 8   TROUBLE_CODES                49739 non-null  object 
 9   TIMING_ADVANCE               49739 non-null  float64
 10  HOURS                        49739 non-null  float64
dtypes: float64(10), object(1)
memory usage: 4.6+ MB


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pickle

class TroubleCodeClassifier:
    def __init__(self):
        # Define the label mapping for trouble codes
        self.label_mapping = {
            'P0133': 0,
            'C0300': 1,
            'P0079P2004P3000': 2,
            'P0078U1004P3000': 3,
            'P0079C1004P3000': 4,
            'P007EP2036P18F0': 5,
            'P007EP2036P18D0': 6,
            'P007FP2036P18D0': 7,
            'P0079P1004P3000': 8,
            'P007EP2036P18E0': 9,
            'P007FP2036P18E0': 10,
            'P0078B0004P3000': 11,
            'P007FP2036P18F0': 12,
            'normal': 13
        }
        self.model = RandomForestClassifier(n_estimators=50, max_depth=7, random_state=42)

    def preprocess_data(self, synthetic_df, data_2):
        """
        Preprocess the datasets by balancing classes and combining them
        """
        size = int(len(synthetic_df)/3)
        normal_indices = synthetic_df[synthetic_df['TROUBLE_CODES'] == 'normal'].sample(n=size, random_state=42, replace = True).index
        synthetic_df = synthetic_df.drop(normal_indices)

        P0133_indices = synthetic_df[synthetic_df['TROUBLE_CODES'] == 'P0133'].sample(n=size+1, random_state=42, replace = True).index
        synthetic_df = synthetic_df.drop(P0133_indices)

        C0300_indices = synthetic_df[synthetic_df['TROUBLE_CODES'] == 'C0300'].sample(n=size - 11, random_state=42, replace = True).index
        synthetic_df = synthetic_df.drop(C0300_indices)

        # Balance the second dataset
        normal_indices = data_2[data_2['TROUBLE_CODES'] == 'normal'].sample(n=35214, random_state=42).index
        data_2 = data_2.drop(normal_indices)

        # Combine datasets
        df = pd.concat([data_2, synthetic_df], ignore_index=True)

        # Map trouble codes to numerical labels
        df['TROUBLE_CODES'] = df['TROUBLE_CODES'].map(self.label_mapping)

        # Prepare features and target
        X = df.drop(columns=['TROUBLE_CODES', 'HOURS'])
        y = df['TROUBLE_CODES']

        return X, y

    def train_model(self, X, y, test_size=0.2):
        """
        Train the model and print evaluation metrics
        """
        # Split the data
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=42)

        # Train the model
        self.model.fit(X_train, y_train)

        # Evaluate and print results
        self._print_evaluation(X_train, y_train, "Training")
        self._print_evaluation(X_test, y_test, "Testing")

        return X_train, X_test, y_train, y_test

    def _print_evaluation(self, X, y, dataset_type):
        """
        Print detailed evaluation metrics
        """
        predictions = self.model.predict(X)
        clf_report = pd.DataFrame(classification_report(y, predictions, output_dict=True))

        print(f"\n{dataset_type} Results:")
        print("=" * 50)
        print(f"Accuracy Score: {accuracy_score(y, predictions) * 100:.2f}%")
        print("\nClassification Report:")
        print(clf_report)

        # Optional: Print feature importances
        if hasattr(self.model, 'feature_importances_'):
            importances = pd.DataFrame({
                'feature': X.columns,
                'importance': self.model.feature_importances_
            }).sort_values('importance', ascending=False)
            print("\nTop 10 Most Important Features:")
            print(importances.head(10))

    def save_model(self, filename='Multi_Classification_Model2.sav'):
        """
        Save the trained model to disk
        """
        pickle.dump(self.model, open(filename, 'wb'))
        print(f"\nModel saved successfully as {filename}")

def main():
    # Initialize the classifier
    classifier = TroubleCodeClassifier()

    # Load your data here
    synthetic_df = combined_data.copy()

    # Preprocess and train
    X, y = classifier.preprocess_data(combined_data, combined_data)

    X_train, X_test, y_train, y_test = classifier.train_model(X, y)

    # Save the model
    classifier.save_model()

if __name__ == "__main__":
    main()


Training Results:
Accuracy Score: 94.92%

Classification Report:
                     0            1           2           3           4  \
precision     0.997124     0.946373    0.368984    0.377358    0.520548   
recall        0.997124     0.942650    0.381215    0.279720    0.260274   
f1-score      0.997124     0.944508    0.375000    0.321285    0.347032   
support    5215.000000  4830.000000  181.000000  143.000000  146.000000   

                    5           6          7          8           9  \
precision    0.597222    0.348837   0.869565   0.677419    0.444444   
recall       0.390909    0.487805   0.217391   0.238636    0.338983   
f1-score     0.472527    0.406780   0.347826   0.352941    0.384615   
support    110.000000  123.000000  92.000000  88.000000  118.000000   

                   10          11         12            13  accuracy  \
precision    0.279693    0.560000   0.833333      0.968510  0.949223   
recall       0.608333    0.277228   0.114943      0.982326

# Vehicle Hours Predictor Model

This demonstrates the development of a regression model to predict the number of hours a vehicle will run based on various features and associated trouble codes.

## Key Components
1. **Model Definition**:
   - An `ExtraTreesRegressor` is used with specific hyperparameters (`max_depth=25`, `n_estimators=30`).
   - A label mapping converts categorical trouble codes into numerical labels for processing.

2. **Data Preprocessing**:
   - Trouble codes are mapped to numerical values.
   - Features (`X`) and the target (`y`) are extracted from the dataset.

3. **Model Training and Evaluation**:
   - The dataset is split into training and testing sets.
   - The model is trained, and metrics such as MAE, MSE, RMSE, and R² are calculated for both sets.
   - Feature importance is analyzed to determine the most influential features.

4. **Model Saving and Loading**:
   - The trained model can be saved to a file and loaded for later use.
---

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pickle

class VehicleHoursPredictor:
    def __init__(self):
        # Define the label mapping for trouble codes
        self.label_mapping = {
            'normal': 0,
            'P0133': 1,
            'C0300': 2,
            'P0079P2004P3000': 3,
            'P0078U1004P3000': 4,
            'P0079C1004P3000': 5,
            'P007EP2036P18F0': 6,
            'P007EP2036P18D0': 7,
            'P007FP2036P18D0': 8,
            'P0079P1004P3000': 9,
            'P007EP2036P18E0': 10,
            'P007FP2036P18E0': 11,
            'P0078B0004P3000': 12,
            'P007FP2036P18F0': 13
        }
        self.model = ExtraTreesRegressor(max_depth=25, n_estimators=30, random_state=42)

    def preprocess_data(self, data):
        """
        Preprocess the dataset for hours prediction
        """
        # Create a copy to avoid modifying original data
        df = data.copy()

        # Map trouble codes to numerical labels
        df['TROUBLE_CODES'] = df['TROUBLE_CODES'].map(self.label_mapping)

        # Prepare features and target
        X = df.drop(columns=['HOURS', 'TROUBLE_CODES'])
        y = df['HOURS']

        return X, y

    def train_model(self, X, y, test_size=0.2):
        """
        Train the model and print evaluation metrics
        """
        # Split the data
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, random_state=42
        )

        # Train the model
        self.model.fit(X_train, y_train)

        # Evaluate and print results
        self._print_evaluation(X_train, y_train, X_test, y_test)

        return X_train, X_test, y_train, y_test

    def _print_evaluation(self, X_train, y_train, X_test, y_test):
        """
        Print detailed evaluation metrics for both training and testing sets
        """
        # Training metrics
        train_pred = self.model.predict(X_train)
        train_metrics = self._calculate_metrics(y_train, train_pred)

        # Testing metrics
        test_pred = self.model.predict(X_test)
        test_metrics = self._calculate_metrics(y_test, test_pred)

        # Print results
        print("\nTraining Results:")
        print("=" * 50)
        self._print_metrics(train_metrics)

        print("\nTesting Results:")
        print("=" * 50)
        self._print_metrics(test_metrics)

        # Print feature importances
        self._print_feature_importance(X_train)

    def _calculate_metrics(self, y_true, y_pred):
        """
        Calculate regression metrics
        """
        return {
            'MAE': mean_absolute_error(y_true, y_pred),
            'MSE': mean_squared_error(y_true, y_pred),
            'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
            'R2': r2_score(y_true, y_pred)
        }

    def _print_metrics(self, metrics):
        """
        Print formatted metrics
        """
        print(f"Mean Absolute Error: {metrics['MAE']:.2f}")
        print(f"Mean Squared Error: {metrics['MSE']:.2f}")
        print(f"Root Mean Squared Error: {metrics['RMSE']:.2f}")
        print(f"R-squared Score: {metrics['R2']:.4f}")

    def _print_feature_importance(self, X):
        """
        Print feature importance analysis
        """
        importances = pd.DataFrame({
            'feature': X.columns,
            'importance': self.model.feature_importances_
        }).sort_values('importance', ascending=False)

        print("\nTop 10 Most Important Features:")
        print(importances.head(10))

    def predict(self, X):
        """
        Make predictions on new data
        """
        return self.model.predict(X)

    def save_model(self, filename='Regression_Hours_Model.sav'):
        """
        Save the trained model to disk
        """
        pickle.dump(self.model, open(filename, 'wb'))
        print(f"\nModel saved successfully as {filename}")

    @staticmethod
    def load_model(filename='Regression_Hours_Model.sav'):
        """
        Load a saved model
        """
        return pickle.load(open(filename, 'rb'))

def main():
    # Initialize the predictor
    predictor = VehicleHoursPredictor()

    # Load your data here
    # data_3 = pd.read_csv('your_data.csv')

    # Preprocess and train
    X, y = predictor.preprocess_data(combined_data)
    X_train, X_test, y_train, y_test = predictor.train_model(X, y)

    # Save the model
    predictor.save_model()

if __name__ == "__main__":
    main()


Training Results:
Mean Absolute Error: 0.77
Mean Squared Error: 1.95
Root Mean Squared Error: 1.40
R-squared Score: 0.9777

Testing Results:
Mean Absolute Error: 2.19
Mean Squared Error: 13.67
Root Mean Squared Error: 3.70
R-squared Score: 0.8445

Top 10 Most Important Features:
                       feature  importance
1          ENGINE_COOLANT_TEMP    0.235878
4              AIR_INTAKE_TEMP    0.219932
3                   ENGINE_RPM    0.205678
0                 ENGINE_POWER    0.085435
2                  ENGINE_LOAD    0.081366
7                 THROTTLE_POS    0.059193
6  SHORT TERM FUEL TRIM BANK 1    0.040553
8               TIMING_ADVANCE    0.036297
5                        SPEED    0.035668

Model saved successfully as Regression_Hours_Model.sav


In [ ]:
combined_data.head()

,ENGINE_POWER,ENGINE_COOLANT_TEMP,ENGINE_LOAD,ENGINE_RPM,AIR_INTAKE_TEMP,SPEED,SHORT TERM FUEL TRIM BANK 1,THROTTLE_POS,TROUBLE_CODES,TIMING_ADVANCE,HOURS
0,1.4,80.0,33.3,1009.0,59.0,0.0,-35.117014,25.0,normal,56.9,16.0
1,1.4,80.0,32.5,1003.0,59.0,0.0,-35.117014,25.0,normal,56.5,16.0
2,1.4,80.0,32.9,995.0,59.0,0.0,-35.117014,25.0,normal,57.3,16.0
3,1.4,80.0,32.5,1004.0,60.0,0.0,-35.117014,25.0,normal,56.5,16.0
4,1.4,80.0,32.9,1005.0,60.0,0.0,-35.117014,25.0,normal,56.9,16.0
